In [1]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
from batch_robot_skill_and_graph import run_batch_robot_skill_and_graph

INTENT_ROOT = "../outputs/intent"
OUT_ROOT = "../outputs"

In [2]:
summary_df, master_graph = run_batch_robot_skill_and_graph(
    intent_root_dir=INTENT_ROOT,
    out_root_dir=OUT_ROOT
)

summary_df.to_csv("../outputs/robot_skill_summary.csv", index=False)
summary_df.head()

Robot skill mapping + graph building: 100%|██████████| 7/7 [00:00<00:00, 373.39it/s]

✅ Exported skill graph: ../outputs/skill_graphs/artificial_jewellery/video_20260404_114752_graph.json
✅ Exported skill graph: ../outputs/skill_graphs/artificial_jewellery/video_20260404_115542_graph.json
✅ Exported skill graph: ../outputs/skill_graphs/artificial_jewellery/video_20260404_120254_graph.json
✅ Exported skill graph: ../outputs/skill_graphs/artificial_jewellery/video_20260404_121005_graph.json
✅ Exported skill graph: ../outputs/skill_graphs/artificial_jewellery/video_20260404_121903_graph.json
✅ Exported skill graph: ../outputs/skill_graphs/artificial_jewellery/video_20260405_105612_edit_graph.json
✅ Exported skill graph: ../outputs/skill_graphs/shop/video_20260405_163219_edit_graph.json

🔗 Building master skill graph with cross-domain links...
✅ Added 0 cross-domain skill transfer edges
✅ Exported skill graph: ../outputs/skill_graphs/DRISHTI_master_graph.json

✅ Master graph nodes: 38
✅ Master graph edges: 31


,domain,video_stem,segments,enriched_csv,graph_json,status
0,artificial_jewellery,video_20260404_114752,4,../outputs/robot_skills/artificial_jewellery/v...,../outputs/skill_graphs/artificial_jewellery/v...,success
1,artificial_jewellery,video_20260404_115542,2,../outputs/robot_skills/artificial_jewellery/v...,../outputs/skill_graphs/artificial_jewellery/v...,success
2,artificial_jewellery,video_20260404_120254,1,../outputs/robot_skills/artificial_jewellery/v...,../outputs/skill_graphs/artificial_jewellery/v...,success
3,artificial_jewellery,video_20260404_121005,2,../outputs/robot_skills/artificial_jewellery/v...,../outputs/skill_graphs/artificial_jewellery/v...,success
4,artificial_jewellery,video_20260404_121903,20,../outputs/robot_skills/artificial_jewellery/v...,../outputs/skill_graphs/artificial_jewellery/v...,success


In [5]:
ok = summary_df[summary_df["status"]=="success"]
if len(ok):
    sample = pd.read_csv(ok.iloc[0]["enriched_csv"])
    
    # Auto-detect available columns from these candidates
    wanted = ["domain", "video_stem", "intent", "intent_confidence",
              "grasp_used", "motion_used", "bimanual",
              "robot_skill", "robot_skill_confidence", "skill_complexity",
              "dof_required", "safety_critical"]
    
    available = [c for c in wanted if c in sample.columns]
    
    display(sample[available].head(15))

,domain,video_stem,intent,intent_confidence,grasp_used,motion_used,bimanual,robot_skill,robot_skill_confidence,skill_complexity,dof_required,safety_critical
0,artificial_jewellery,video_20260404_114752,jewellery_assembling,0.3,partial_grip,fast,True,precision_placement,0.8,medium,5,False
1,artificial_jewellery,video_20260404_114752,jewellery_assembling,0.3,partial_grip,fast,True,precision_placement,0.8,medium,5,False
2,artificial_jewellery,video_20260404_114752,jewellery_sorting,0.2,open_hand,fast,False,precision_sorting,0.9,medium,5,False
3,artificial_jewellery,video_20260404_114752,jewellery_assembling,0.3,partial_grip,fast,True,precision_placement,0.8,medium,5,False


In [6]:
print("Master Graph Stats:")
print("Nodes:", master_graph.number_of_nodes())
print("Edges:", master_graph.number_of_edges())

# Count edge types
edge_types = {}
for u, v, attrs in master_graph.edges(data=True):
    et = attrs.get("edge_type", "unknown")
    edge_types[et] = edge_types.get(et, 0) + 1

print("\nEdge types:")
for et, count in edge_types.items():
    print(f"  {et}: {count}")

Master Graph Stats:
Nodes: 38
Edges: 31

Edge types:
  temporal_next: 31


In [7]:
all_enriched = []
for _, row in ok.iterrows():
    df = pd.read_csv(row["enriched_csv"])
    all_enriched.append(df)

if all_enriched:
    combined = pd.concat(all_enriched, ignore_index=True)
    print("Robot Skill Distribution:")
    print(combined["robot_skill"].value_counts())
    
    print("\nSkill Complexity:")
    print(combined["skill_complexity"].value_counts())

Robot Skill Distribution:
robot_skill
dexterous_fine_manipulation    19
precision_sorting               7
precision_placement             4
handover_interaction            1
Name: count, dtype: int64

Skill Complexity:
skill_complexity
high      19
medium    12
Name: count, dtype: int64
